# 07 — Model Preprocessing & Leakage Control

**Người phụ trách:** Chu Minh Tuấn — Model Data Lead  
**Mục tiêu Bài tập 4 / Phần D.1:** chuẩn bị dữ liệu bằng `scikit-learn Pipeline`, kiểm soát data leakage và xử lý đa cộng tuyến có cơ sở.

Notebook này **không huấn luyện/chọn mô hình cuối**. Nó khóa “data contract” để Bùi Đức Mạnh có thể đưa Linear/Ridge/RandomForest/... vào cùng một pipeline và cross-validation mà không làm rò rỉ dữ liệu.

### Quy tắc bắt buộc
1. Target `price_per_m2` chỉ được tạo từ `gia_goc / dien_tich_goc` khi **cả hai thực sự quan sát được**; không dùng giá/diện tích đã impute để tạo nhãn.
2. Khử trùng theo `link_nguon` trước khi split.
3. Theo Project Charter: chỉ tin **rao bán**, diện tích **10–2.000 m²**; tin cho thuê ngoài scope.
4. **Split 80/20 trước** mọi bước học tham số (median, scale, category encoding).
5. Không đưa `gia`, `gia_goc`, `price_per_m2` vào X.
6. Không dùng các cột one-hot đã được tạo trên toàn bộ `cafeland_features.csv`; one-hot phải được fit lại bên trong Pipeline trên train/CV fold.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

here = Path.cwd().resolve()
project_root = here.parent if here.name == 'notebooks' else here
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.modeling.preprocess import (
    TARGET, MODEL_FEATURES,
    build_modeling_table, split_model_data, build_preprocessor,
    training_target_quantile_diagnostics, correlation_report, vif_report,
    feature_contract, save_split_tables,
)

SOURCE = project_root / 'data' / 'processed' / 'cafeland_missing_outliers_cleaned.csv'
REPORTS = project_root / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
print('Project root:', project_root)
print('Source:', SOURCE)

Project root: /mnt/data/project_bt4/Final_Project_Nhom_22-24-main
Source: /mnt/data/project_bt4/Final_Project_Nhom_22-24-main/data/processed/cafeland_missing_outliers_cleaned.csv


## 1. Dựng modeling base từ dữ liệu có provenance

Ta quay lại `cafeland_missing_outliers_cleaned.csv` vì file này còn `link_nguon`, `gia_goc`, `dien_tich_goc` và các cờ thiếu. Đây là những trường cần thiết để khử trùng và chứng minh target không được tạo từ dữ liệu đã impute.

In [2]:
source = pd.read_csv(SOURCE, encoding='utf-8-sig')
modeling, audit = build_modeling_table(source)

display(audit)
print(f'\nModeling base: {len(modeling):,} dòng x {modeling.shape[1]} cột')
display(modeling.head(3))

,step,rows_before,rows_after,rows_removed,removed_pct_of_step,reason
0,01_deduplicate_link,5435,5388,47,0.865,Mỗi link_nguon là một tin; giữ bản ghi đầu tiê...
1,02_observed_target_only,5388,3564,1824,33.853,Target dùng gia_goc/dien_tich_goc quan sát thự...
2,03_area_scope_10_2000,3564,3558,6,0.168,Theo Project Charter: chỉ giữ diện tích từ 10 ...
3,04_sale_scope,3558,3480,78,2.192,Project Charter đặt tin cho thuê ngoài phạm vi...
4,05_positive_finite_target,3480,3480,0,0.000,Giữ price_per_m2 dương và hữu hạn; không học n...



Modeling base: 3,480 dòng x 9 cột


,link_nguon,tieu_de,vi_tri,gia_goc,dien_tich_goc,so_phong_ngu,khu_vuc_nhom,loai_bds,price_per_m2
0,https://nhadat.cafeland.vn/ket-tien-can-ban-2p...,KẸT TIỀN CẦN BÁN 2PN+ 72M TẦNG 20+ MAISON GRAN...,"Phú Mỹ, Thành phố Hồ Chí Minh",2.990000e+09,72.0,2.0,TP.HCM (lõi cũ),Khác,4.152778e+07
1,https://nhadat.cafeland.vn/ban-nha-moi-keng-ma...,"BÁN NHÀ MỚI KENG MẶT TIỀN TÂN XUÂN, HÓC MÔN ,S...","An Lạc, Thành phố Hồ Chí Minh",5.350000e+09,100.0,3.0,TP.HCM (lõi cũ),Nhà ở/Nhà phố,5.350000e+07
2,https://nhadat.cafeland.vn/dat-1nguyen-thi-san...,"ĐẤT 1/NGUYỄN THỊ SÁNG , ĐÔNG THẠNH ,HÓC MÔN 5M...","Chợ Lớn, Thành phố Hồ Chí Minh",3.750000e+09,105.0,NaN,TP.HCM (lõi cũ),Đất nền,3.571429e+07


In [3]:
# Các kiểm tra khóa target và scope
assert modeling['link_nguon'].is_unique
assert modeling[TARGET].notna().all()
assert (modeling[TARGET] > 0).all()
assert modeling['dien_tich_goc'].between(10, 2000).all()

print('Target price_per_m2 (VNĐ/m²):')
display(modeling[TARGET].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).to_frame().T)
print('\nThiếu feature:')
display((modeling[MODEL_FEATURES].isna().mean()*100).round(2).rename('% thiếu').to_frame())

Target price_per_m2 (VNĐ/m²):


,count,mean,std,min,1%,5%,50%,95%,99%,max
price_per_m2,3480.0,1.197301e+08,9.124232e+07,0.010579,4.320929e+06,1.526996e+07,100000000.0,2.840927e+08,4.603231e+08,8.290155e+08



Thiếu feature:


,% thiếu
dien_tich_goc,0.00
so_phong_ngu,60.95
khu_vuc_nhom,0.00
loai_bds,0.00


### Ghi chú về target cực trị

Project Charter có nêu hướng xử lý **cắt phân vị 1%–99% giá/m²**. Tuy nhiên, để không dùng thông tin của hold-out test khi đặt ngưỡng, notebook này **không tính percentile trên toàn bộ dữ liệu**. Sau khi split, ta chỉ ước lượng ngưỡng trên `y_train` để chẩn đoán. Hold-out test vẫn được giữ nguyên; mọi quyết định loại/giữ target cực trị khi tuning phải được ghi rõ và không được “nhìn test rồi sửa”.

## 2. Train/test split trước preprocessing

Từ đây trở đi, mọi thống kê học từ dữ liệu (`median`, `mean/std`, category dictionary) chỉ được fit trên train hoặc từng fold của cross-validation.

In [4]:
split = split_model_data(modeling, test_size=0.20, random_state=42)

print(f'Train: {len(split.X_train):,} dòng ({len(split.X_train)/len(modeling):.1%})')
print(f'Test : {len(split.X_test):,} dòng ({len(split.X_test)/len(modeling):.1%})')
print('Features X:', list(split.X_train.columns))

assert set(split.X_train.columns) == set(MODEL_FEATURES)
assert 'gia' not in split.X_train.columns
assert 'gia_goc' not in split.X_train.columns
assert TARGET not in split.X_train.columns

train_links = set(modeling.loc[split.train_index, 'link_nguon'])
test_links = set(modeling.loc[split.test_index, 'link_nguon'])
assert train_links.isdisjoint(test_links)
print('✅ Không có link_nguon trùng giữa train/test.')

Train: 2,784 dòng (80.0%)
Test : 696 dòng (20.0%)
Features X: ['dien_tich_goc', 'so_phong_ngu', 'khu_vuc_nhom', 'loai_bds']
✅ Không có link_nguon trùng giữa train/test.


## 3. ColumnTransformer + Pipeline preprocessing

- Numeric: median imputation **fit trên train** + missing indicator + `StandardScaler`.
- Categorical: most-frequent imputation + `OneHotEncoder(handle_unknown="ignore")`.
- Reference category cố định: `TP.HCM (lõi cũ)` và `Nhà ở/Nhà phố`. Đây là hai nhóm nền dễ diễn giải và giúp giảm đa cộng tuyến so với việc vô tình dùng một nhóm hiếm làm baseline.

**Quan trọng cho Modeling Lead:** không lấy ma trận đã transform ở notebook này để chạy cross-validation. Hãy gọi `build_model_pipeline(estimator)` để preprocessor được refit trong **từng CV fold**.

In [5]:
preprocessor = build_preprocessor()
X_train_tx = preprocessor.fit_transform(split.X_train)  # chỉ fit TRAIN để diagnostics
X_test_tx = preprocessor.transform(split.X_test)        # test chỉ transform
feature_names = preprocessor.get_feature_names_out()

print('Shape train sau transform:', X_train_tx.shape)
print('Shape test  sau transform:', X_test_tx.shape)
print('Feature sau transform:')
for name in feature_names:
    print(' -', name)

assert np.isfinite(X_train_tx).all()
assert np.isfinite(X_test_tx).all()
print('✅ Không còn missing/inf sau preprocessing.')

Shape train sau transform: (2784, 9)
Shape test  sau transform: (696, 9)
Feature sau transform:
 - num__dien_tich_goc
 - num__so_phong_ngu
 - num__missingindicator_so_phong_ngu
 - cat__khu_vuc_nhom_Bà Rịa - Vũng Tàu (mới sáp nhập)
 - cat__khu_vuc_nhom_Bình Dương (mới sáp nhập)
 - cat__loai_bds_Biệt thự
 - cat__loai_bds_Căn hộ/Chung cư
 - cat__loai_bds_Khác
 - cat__loai_bds_Đất nền
✅ Không còn missing/inf sau preprocessing.


## 4. Correlation & VIF — chỉ trên TRAIN

VIF được tính trên thiết kế đã impute/scale/one-hot bằng preprocessor **fit trên train**. Mức VIF < 5 được xem là thấp trong notebook này; mục tiêu là chứng minh không có đa cộng tuyến nghiêm trọng do encoding.

In [6]:
train_for_corr = modeling.loc[split.train_index]
corr = correlation_report(train_for_corr)
vif = vif_report(preprocessor, split.X_train)

display(corr.round(3))
display(vif.round(3))

print('VIF lớn nhất:', round(vif['VIF'].max(), 3))
assert vif['VIF'].max() < 5, 'Cần xem lại đa cộng tuyến: có VIF >= 5.'
print('✅ VIF < 5 cho toàn bộ design matrix hiện tại.')

,dien_tich_goc,so_phong_ngu,price_per_m2
dien_tich_goc,1.000,0.411,-0.187
so_phong_ngu,0.411,1.000,0.204
price_per_m2,-0.187,0.204,1.000


,feature,VIF,assessment
0,num__dien_tich_goc,1.219,thấp
1,num__missingindicator_so_phong_ngu,1.207,thấp
2,cat__loai_bds_Căn hộ/Chung cư,1.205,thấp
3,cat__loai_bds_Đất nền,1.196,thấp
4,num__so_phong_ngu,1.110,thấp
5,cat__loai_bds_Khác,1.101,thấp
6,cat__loai_bds_Biệt thự,1.097,thấp
7,cat__khu_vuc_nhom_Bình Dương (mới sáp nhập),1.097,thấp
8,cat__khu_vuc_nhom_Bà Rịa - Vũng Tàu (mới sáp n...,1.047,thấp


VIF lớn nhất: 1.219
✅ VIF < 5 cho toàn bộ design matrix hiện tại.


## 5. Chẩn đoán percentile target trên TRAIN chỉ

Đây là **diagnostic**, không tự động xóa test. Ngưỡng được fit trên train để tránh rò rỉ phân phối target của test. Nếu nhóm muốn thực hiện cắt 1%–99% đúng Project Charter, cần mô tả rõ cách áp dụng trong quy trình đánh giá; tuyệt đối không chọn ngưỡng sau khi nhìn metric test.

In [7]:
target_diag, target_extreme_mask = training_target_quantile_diagnostics(split.y_train)
display(target_diag)

extremes = modeling.loc[split.y_train.index[target_extreme_mask], [
    'link_nguon', 'tieu_de', 'gia_goc', 'dien_tich_goc', TARGET
]].sort_values(TARGET)
print(f'Số target train bị gắn cờ ngoài P1–P99: {len(extremes)}')
display(extremes.head(5))
display(extremes.tail(5))

,lower_quantile,upper_quantile,lower_bound_vnd_m2,upper_bound_vnd_m2,train_rows,flagged_train_rows,flagged_train_pct,policy
0,0.01,0.99,4.388667e+06,4.630769e+08,2784,56,2.011,diagnostic_only_train_fitted; hold-out test un...


Số target train bị gắn cờ ngoài P1–P99: 56


,link_nguon,tieu_de,gia_goc,dien_tich_goc,price_per_m2
285,https://nhadat.cafeland.vn/dat-mat-tien-duong-...,"Đất mặt tiền đường nhựa DH606 , Thành Phố Hồ C...",1.48,139.9,0.010579
1790,https://nhadat.cafeland.vn/ngop-ngan-hang-can-...,"Ngộp ngân hàng, cần ra gấp căn nhà Tân Bình gi...",2.25,62.0,0.036290
931,https://nhadat.cafeland.vn/can-ho-70m2-ban-gia...,Căn Hộ 70m2 bàn giao cao cấp giá đang sale chỉ...,29.99,70.0,0.428429
453,https://nhadat.cafeland.vn/gh-doc-quyen-lusso-...,"GH Độc Quyền LUSSO SAIGON Ngay Ga Metro, Mở Rộ...",2400000.00,56.0,42857.142857
1485,https://nhadat.cafeland.vn/chuyen-nhuong-nha-m...,Chuyển nhượng nhà mặt tiền đường Nguyễn Đình C...,17000000.00,56.0,303571.428571


,link_nguon,tieu_de,gia_goc,dien_tich_goc,price_per_m2
61,https://nhadat.cafeland.vn/top-nha-vi-tri-dep-...,Top nhà vị trí đẹp 4 x 14.5m 1 trệt 3 lầu Cô G...,3.250000e+10,56.00,5.803571e+08
68,https://nhadat.cafeland.vn/nha-vi-tri-dep-4-x-...,Nhà vị trí đẹp 4 x 15m Nguyễn Văn Thủ trung tâ...,3.500000e+10,60.00,5.833333e+08
3187,https://nhadat.cafeland.vn/ban-nha-rieng-ban-n...,Bán nhà riêng: Bán nhà MT 193 Cô Giang P Cầu Ô...,2.500000e+10,42.00,5.952381e+08
1248,https://nhadat.cafeland.vn/ban-nha-mat-tien-du...,"Bán nhà mặt tiền đường Nguyễn Công Trứ, phường...",2.300000e+10,35.63,6.455234e+08
1660,https://nhadat.cafeland.vn/ban-nha-mat-tien-du...,"Bán nhà mặt tiền đường 3/2, Phường 11, Quận 10...",1.450000e+10,18.00,8.055556e+08


## 6. Feature contract: được phép / cấm đưa vào X

Bảng này là “hợp đồng bàn giao” giữa Tuấn và Mạnh. Nếu thêm feature mới, phải cập nhật bảng và kiểm tra lại leakage/VIF.

In [8]:
contract = feature_contract()
display(contract)

,column,status,type,reason
0,dien_tich_goc,ALLOWED,Numeric,Diện tích quan sát thực; dùng làm feature và m...
1,so_phong_ngu,ALLOWED,Numeric,Có thiếu; median + missing indicator được fit ...
2,khu_vuc_nhom,ALLOWED,Categorical,Nhóm địa lý compact từ Bài tập 3; OneHotEncode...
3,loai_bds,ALLOWED,Categorical,Loại BĐS suy ra từ tiêu đề; OneHotEncoder fit ...
4,gia,FORBIDDEN,Leakage,Giá sau impute; target price_per_m2 = giá/diện...
5,gia_goc,FORBIDDEN,Leakage,Tử số trực tiếp của target; tuyệt đối không đư...
6,price_per_m2,FORBIDDEN,Target,"Biến mục tiêu y, không phải feature."
7,gia_thieu,FORBIDDEN,Constant after scope,"Sau observed-target filter cờ này luôn False, ..."
8,dien_tich_thieu,FORBIDDEN,Constant after scope,"Sau observed-target filter cờ này luôn False, ..."
9,tieu_de,FORBIDDEN_CURRENT,Free text,Chỉ dùng để suy ra loai_bds; chưa có NLP pipel...


## 7. Lưu sản phẩm bàn giao

Các file `model_train_raw.csv` / `model_test_raw.csv` vẫn là dữ liệu **chưa fit preprocessing**. Đây là chủ đích: Modeling Lead phải để `build_preprocessor()` bên trong Pipeline để cross-validation không bị leakage.

In [9]:
base_path, train_path, test_path = save_split_tables(
    modeling, split, project_root / 'data' / 'processed'
)

audit.to_csv(REPORTS / 'preprocessing_scope_audit.csv', index=False, encoding='utf-8-sig')
corr.to_csv(REPORTS / 'correlation_train.csv', encoding='utf-8-sig')
vif.to_csv(REPORTS / 'vif_train.csv', index=False, encoding='utf-8-sig')
contract.to_csv(REPORTS / 'feature_contract.csv', index=False, encoding='utf-8-sig')
target_diag.to_csv(REPORTS / 'target_quantile_train_diagnostic.csv', index=False, encoding='utf-8-sig')

print('Đã lưu:')
for path in [
    base_path, train_path, test_path,
    REPORTS / 'preprocessing_scope_audit.csv',
    REPORTS / 'correlation_train.csv',
    REPORTS / 'vif_train.csv',
    REPORTS / 'feature_contract.csv',
    REPORTS / 'target_quantile_train_diagnostic.csv',
]:
    print(' -', path.relative_to(project_root))

Đã lưu:
 - data/processed/modeling_base.csv
 - data/processed/model_train_raw.csv
 - data/processed/model_test_raw.csv
 - reports/preprocessing_scope_audit.csv
 - reports/correlation_train.csv
 - reports/vif_train.csv
 - reports/feature_contract.csv
 - reports/target_quantile_train_diagnostic.csv


## 8. Kết luận bàn giao

- Target được tạo từ **giá/diện tích gốc có quan sát thực**, không dùng giá/diện tích đã impute.
- Đã khử duplicate theo `link_nguon`, lọc scope đúng Project Charter và khóa split 80/20 với `random_state=42`.
- X chỉ gồm: `dien_tich_goc`, `so_phong_ngu`, `khu_vuc_nhom`, `loai_bds`.
- `gia`, `gia_goc` và target bị cấm tuyệt đối trong X.
- Median imputation, missing indicator, scaling, one-hot đều nằm trong `ColumnTransformer`; downstream phải đặt transformer trong model Pipeline để fit lại theo từng CV fold.
- Correlation/VIF được kiểm tra trên **train**, không dựa vào test.
- Các target ngoài P1–P99 chỉ được gắn cờ chẩn đoán bằng ngưỡng học từ train; hold-out test không bị dùng để đặt ngưỡng.

**Bàn giao cho Bùi Đức Mạnh:** import `build_model_pipeline` từ `src.modeling.preprocess`, truyền estimator vào rồi chạy cross-validation/tuning trên train. Test chỉ dùng một lần ở đánh giá cuối.